In [2]:
!pip install dgllife

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.1/226.1 kB 5.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 kB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 63.6 MB/s eta 0:00:00


In [6]:
import dgl
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from dgllife.model.model_zoo import GCNPredictor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import add_hydrogen_bond_interactions, add_peptide_bonds, add_k_nn_edges
from graphein.protein.graphs import construct_graph
from graphein.protein.utils import download_pdb

from matplotlib import pyplot as plt
device = 'cpu'

In [5]:
df = pd.read_csv('structural_rearrangement_data.csv')
df.head()

,Unnamed: 0,level_0,index,PSCID,Protein Name,Free form,Bound form,Ligands,Classification(?),motion_type,Free PDB,Free Chains,Bound PDB,Bound Chains
0,0,0,0,CD.1,HYPOTHETICAL OXIDOREDUCTASE YIAK,1nxu_AB,1s20_AB,"2xNAD,2xTLA",200004,coupled_domain_motion,1nxu,AB,1s20,AB
1,1,1,1,CD.2,ADENYLATE KINASE,4ake_A,2eck_A,"ADP,AMP",200003,coupled_domain_motion,4ake,A,2eck,A
2,2,2,2,CD.3,GLUCOKINASE,1q18_AB,1sz2_AB,2xBGC,200003,coupled_domain_motion,1q18,AB,1sz2,AB
3,3,3,3,CD.4,LACTOFERRIN,1lfh_A,1lfi_A,"2xCU,2xNAG",110103,coupled_domain_motion,1lfh,A,1lfi,A
4,4,4,4,CD.5,ELONGATION FACTOR 2,1n0v_D,1n0u_A,SO1,110002,coupled_domain_motion,1n0v,D,1n0u,A


In [14]:
# config = ProteinGraphConfig([granularity='CA', insertions=False, keep_hets=True,
#                   node_featuriser='meiler', get_contacts_path='/Users/arianjamasb/github/getcontacts',
#                   pdb_dir='../../examples/pdbs/',
#                   contacts_dir='../../examples/contacts/',
#                   exclude_waters=True, covalent_bonds=False, include_ss=True])

params_to_change = {'granularity': 'CA', 'insertions': False, 'keep_hets': True, 'edge_cosntructions_functions': [add_peptide_bonds]}

config = ProteinGraphConfig(**params_to_change)

 

ValidationError: 1 validation error for ProteinGraphConfig
keep_hets
  Input should be a valid list [type=list_type, input_value=True, input_type=bool]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type